In [218]:
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from abc import abstractmethod
from collections import Counter
import numpy as np
import math
import os

In [219]:
def load_data(folder):
    x_train = np.load(os.path.join(folder, 'x_train.npy'))
    y_train = np.load(os.path.join(folder, 'y_train.npy'))
    x_test = np.load(os.path.join(folder, 'x_test.npy'))
    y_test = np.load(os.path.join(folder, 'y_test.npy'))
    return x_train, y_train, x_test, y_test

In [220]:
def assert_preds_correct(your_preds, sklearn_preds) -> bool:
    return np.abs(your_preds - sklearn_preds).sum() == 0

In [221]:
def assert_probs_correct(your_probs, sklearn_probs) -> bool:
    return np.abs(your_probs - sklearn_probs).mean() < 1e-3

In [222]:
# Не изменяйте код этого класса!
class NaiveBayes:
    def __init__(self, n_classes):
        self.n_classes = n_classes
        self.params = dict()

    # --- PREDICTION ---

    def predict(self, x, return_probs=False):
        """
        x - np.array размерности [N, dim],
        где N - количество экземпляров данных,
        dim -размерность одного экземпляра (количество признаков).

        Возвращает np.array размерности [N], содержащий номера классов для
        соответствующих экземпляров.
        """
        preds = []
        for sample in x:
            preds.append(
                self.predict_single(sample, return_probs=return_probs)
            )

        if return_probs:
            return np.array(preds, dtype='float32')

        return np.array(preds, dtype='int32')

    # Совет: вниманительно изучите файл подсказок к данной лабораторной
    # и сопоставьте код с описанной математикой байесовского классификатора.
    def predict_single(self, x, return_probs=False) -> int:
        """
        Делает предсказание для одного экземпляра данных.

        x - np.array размерности dim.

        Возвращает номер класса, которому принадлежит x.
        """
        assert len(x.shape) == 1, f'Expected a vector, but received a tensor of shape={x.shape}'
        marginal_prob = self.compute_marginal_probability(x)  # P(x) - безусловная вероятность появления x

        probs = []
        for c in range(self.n_classes):                 # c - номер класса
            prior = self.compute_prior(c)               # P(c) - априорная вероятность (вероятность появления класса)
            likelihood = self.compute_likelihood(x, c)  # P(x|c) - вероятность появления x в предположении, что он принаждлежит c

            # Используем теорему Байесса для просчёта условной вероятности P(c|x)
            # P(c|x) = P(c) * P(x|c) / P(x)
            prob = prior * likelihood / marginal_prob
            probs.append(prob)

        if return_probs:
            return probs

        return np.argmax(probs)

    # Вычисляет P(x) - безусловная вероятность появления x.
    @abstractmethod
    def compute_marginal_probability(self, x) -> float:
        pass

    # Вычисляет P(c) - априорная вероятность появления класса c.
    @abstractmethod
    def compute_prior(self, c) -> float:
        pass

    # Вычисляет P(x|c) - вероятность наблюдения экземпляра x в предположении, что он принаждлежит c.
    @abstractmethod
    def compute_likelihood(self, x, c) -> float:
        pass

    # --- FITTING ---

    def fit(self, x, y):
        self._estimate_prior(y)
        self._estimate_params(x, y)

    @abstractmethod
    def _estimate_prior(self, y):
        pass

    @abstractmethod
    def _estimate_params(self, x, y):
        pass

Вынесла отдельно функцию для расчета и вывода метрик, чтобы избежать дублирования кода

In [223]:
def print_classification_metrics(y_true, y_pred, title=""):
    """
    Выводит метрики качества классификации (F1-мера, Полнота, Точность).

    Args:
        y_true (array-like): Истинные метки классов.
        y_pred (array-like): Предсказанные метки классов.
        title (str, optional): Заголовок для вывода метрик.
    """
    if title:
        print(f"\n--- {title} ---")

    # Расчет метрик
    f1 = f1_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    precision = precision_score(y_true, y_pred, average='weighted')

    # Вывод результатов
    print(f"F1-мера: {f1:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"Precision: {precision:.4f}")

## 1. Наивный классификатор Байеса: гауссово распределение

Напишите недостающий код, создайте и обучите модель.

Пункты оценки:
1. совпадение предсказанных классов с оными у модели sklearn. Для проверки совпадения используйте функцию `assert_preds_correct`.
2. совпадение значений предсказанных вероятностей принадлежности классами с оными у модели sklearn. Значения вероятностей считаются равными, если функция `assert_probs_correct` возвращает True.

In [224]:
x_train, y_train, x_test, y_test = load_data('gauss')

In [225]:
class NaiveGauss(NaiveBayes):
    """
    Этот класс реализует алгоритм Гауссова наивного Байеса.
    """

    def compute_marginal_probability(self, x: np.ndarray) -> float:
        """
        Вычисляет безусловную вероятность P(x) для заданного вектора
        признаков x.

        P(x) = sum_c (P(c) * P(x|c))

        Args:
            x (np.ndarray): Одномерный массив признаков, для которого
                            вычисляется безусловная вероятность.

        Returns:
            float: Вычисленная безусловная вероятность P(x).
        """
        marginal_prob = 0.0
        for c in range(self.n_classes):
            prior = self.compute_prior(c)
            likelihood = self.compute_likelihood(x, c)
            marginal_prob += prior * likelihood
        return marginal_prob

    def compute_prior(self, c: int) -> float:
        """
        Вычисляет априорную вероятность P(c) для заданного класса c.

        Args:
            c (int): Индекс класса, для которого вычисляется априорная
                     вероятность.

        Returns:
            float: Априорная вероятность P(c).
        """
        return self.params['prior'][c]

    def compute_likelihood(self, x: np.ndarray, c: int) -> float:
        """
        Вычисляет вероятность правдоподобия P(x|c) для заданного вектора
        признаков x и класса c, предполагая гауссово распределение для
        каждого признака.

        P(x|c) = product_j(P(x_j | c)), где P(x_j | c) - это функция
        плотности вероятности Гаусса для признака j.

        Args:
            x (np.ndarray): Одномерный массив признаков.
            c (int): Индекс класса.

        Returns:
            float: Вычисленная вероятность правдоподобия P(x|c).
        """
        mean = self.params['mean'][c]
        var = self.params['var'][c]

        # Гауссова функция плотности вероятности для каждого признака:
        # P(x_j | c) = 1 / sqrt(2*pi*var_j) * exp(-(x_j - mean_j)^2 /
        # (2*var_j))
        # P(x | c) = product_j(P(x_j | c))

        # Чтобы избежать деления на ноль при var = 0, добавляем очень
        # маленькое число.
        epsilon = 1e-9
        var_smoothed = var + epsilon

        exponent_term = -((x - mean) ** 2) / (2 * var_smoothed)
        gaussian_term = np.exp(exponent_term) / (
            np.sqrt(2 * np.pi * var_smoothed))

        return np.prod(gaussian_term)

    # --- FITTING ---

    def _estimate_prior(self, y: np.ndarray):
        """
        Оценивает априорные вероятности для каждого класса на основе
        обучающих меток.

        P(c) = (Количество образцов в классе c) / (Общее количество образцов)

        Args:
            y (np.ndarray): Обучающие метки.
        """
        prior = np.zeros(self.n_classes)
        for c in range(self.n_classes):
            prior[c] = np.sum(y == c) / len(y)
        self.params['prior'] = prior

    def _estimate_params(self, x: np.ndarray, y: np.ndarray):
        """
        Оценивает среднее значение (mean) и дисперсию (variance) для
        каждого признака по каждому классу.

        Args:
            x (np.ndarray): Обучающие данные.
            y (np.ndarray): Обучающие метки.
        """
        dim = x.shape[1]
        mean = np.zeros((self.n_classes, dim))
        var = np.zeros((self.n_classes, dim))

        for c in range(self.n_classes):
            x_c = x[y == c]
            mean[c] = np.mean(x_c, axis=0)
            var[c] = np.var(x_c, axis=0)

        self.params['mean'] = mean
        self.params['var'] = var

In [226]:
# Определяем количество классов динамически
n_classes_from_data = len(np.unique(y_train))
print("Количество классов:", n_classes_from_data)

Количество классов: 3


In [227]:
# Создайте и обучите модель
# Модель NaiveGauss собственная
model = NaiveGauss(n_classes=n_classes_from_data)
model.fit(x_train, y_train)
print("Собственная модель NaiveGauss обучена.")

# Модель GaussianNB из библиотеки scikit-learn
sklearn_model = GaussianNB()
sklearn_model.fit(x_train, y_train)
print("Модель GaussianNB из scikit-learn обучена.")

Собственная модель NaiveGauss обучена.
Модель GaussianNB из scikit-learn обучена.


In [228]:
# Оцените качество модели
# Получаем предсказания и вероятности от собственной модели
preds = model.predict(x_test)
probs = model.predict(x_test, return_probs=True)

# Получаем предсказания и вероятности от модели из библиотеки scikit-learn
sklearn_preds = sklearn_model.predict(x_test)
sklearn_probs = sklearn_model.predict_proba(x_test)

print_classification_metrics(
    y_test, preds, "Метрики качества собственной модели NaiveGauss"
)
print_classification_metrics(
    y_test, sklearn_preds, "Метрики качества модели Sklearn GaussianNB"
)


--- Метрики качества собственной модели NaiveGauss ---
F1-мера: 1.0000
Recall: 1.0000
Precision: 1.0000

--- Метрики качества модели Sklearn GaussianNB ---
F1-мера: 1.0000
Recall: 1.0000
Precision: 1.0000


In [229]:
# Сравните вашу модель с аналогом sklearn (GaussianNB)
preds_gaussian_ok = assert_preds_correct(preds, sklearn_preds)
print("Проверка совпадения предсказаний:", preds_gaussian_ok)

probs_gaussian_ok = assert_probs_correct(probs, sklearn_probs)
print("Проверка совпадения вероятностей:", probs_gaussian_ok)

Проверка совпадения предсказаний: True
Проверка совпадения вероятностей: True


In [230]:
# Делаем окончательный вывод на основе всех результатов
if preds_gaussian_ok and probs_gaussian_ok:
    print("\nВсе проверки для Gaussian Naive Bayes пройдены успешно!")
else:
    print("\nНекоторые проверки для Gaussian Naive Bayes НЕ пройдены.")
    if not preds_gaussian_ok:
        print("  - Проверка предсказаний НЕ пройдена.")
    if not probs_gaussian_ok:
        print("  - Проверка вероятностей НЕ пройдена.")


Все проверки для Gaussian Naive Bayes пройдены успешно!


## 2. Доп. задания (любое на выбор, опционально)

### 2.1  Упрощение наивного классификатора Байеса для гауссова распределения

Уберите из класса NaiveBayes 'лишние' вычисления и удалите код, что соответствует этим вычислениям. Под 'лишним' подразумеваются вещи, что не влияют на итоговое решение о принадлежности классу (значения вероятностей при этом могу стать некорректными, но в данном задании это допустимо).

Напишите в клетке ниже код упрощенного 'классификатора Гаусса' и убедитесь, что его ответы (не значения вероятностей) совпадают с ответами классификатора из задания 1. Для сравнения ответов используйте функцию `assert_preds_correct`.

Указание: работайте в предположении, что классы равновероятны.

Подсказка: упростить необходимо метод `predict_single`.

In [231]:
class SimplifiedNaiveGauss(NaiveGauss):
    """
    Упрощенный наивный классификатор Байеса для гауссова распределения.
    """

    def predict_single(self, x, return_probs=False) -> int:
        """Предсказывает класс для вектора признаков.

        Вычисляет оценки (правдоподобия) для каждого класса.
        Этого достаточно для классификации, так как P(x) и P(c) игнорируются.

        Args:
            x (numpy.ndarray или list): Входной вектор.
            return_probs (bool): Если True, возвращает список оценок.

        Returns:
            int: Предсказанная метка класса.
            list[float]: Список оценок классов, если return_probs=True.
        """
        scores = []
        for c in range(self.n_classes):
            likelihood = self.compute_likelihood(x, c)
            scores.append(likelihood)

        if return_probs:
            return scores

        return np.argmax(scores)

In [232]:
# Создайте и обучите упрощенную модель
simplified_model = SimplifiedNaiveGauss(n_classes=n_classes_from_data)
simplified_model.fit(x_train, y_train)
print("Упрощенная модель SimplifiedNaiveGauss обучена.")

Упрощенная модель SimplifiedNaiveGauss обучена.


In [233]:
# Оцените качество модели
simplified_preds = simplified_model.predict(x_test)
print_classification_metrics(
    y_test,
    simplified_preds,
    "Метрики качества упрощенной модели SimplifiedNaiveGauss"
)


--- Метрики качества упрощенной модели SimplifiedNaiveGauss ---
F1-мера: 1.0000
Recall: 1.0000
Precision: 1.0000


In [234]:
# Сравним предсказания упрощенной модели с оригинальной
print(
    "Предсказания упрощенной модели совпадают с "
    "оригинальной NaiveGauss:",
    assert_preds_correct(simplified_preds, preds)
)

Предсказания упрощенной модели совпадают с оригинальной NaiveGauss: True


In [235]:
# Объясните в комментариях к этой клетке суть проделанных изменений:
# почему удаленный код является лишним?

# Упрощение основано на том, что для определения класса (т.е. нахождения
# argmax по P(c|x)) достаточно сравнить только значения функции
# правдоподобия P(x|c). Упрощение достигается за счет исключения
# константных членов в формуле Байеса:
# Причина:
# 1. Вероятность P(x) (безусловная вероятность появления x) является
# константой для данного экземпляра x и не влияет на результат argmax
# по классам c.
# 2. Согласно условию "работайте в предположении, что классы
# равновероятны", априорная вероятность P(c) является константой для
# всех классов c. Умножение на такую константу также не влияет на
# результат argmax.

# Таким образом, задача поиска argmax_c(P(c|x)) сводится к поиску
# argmax_c(P(x|c)).

### 2.1  Наивный классификатор Байеса: мультиномиальное распределения

Напишите недостающий код, создайте и обучите модель.

Подсказка: в определении функции правдоподобия много факториалов. Для избежания численного переполнения посчитайте сначала логарифм функции правдоподобия (на бумаге), после примените экспоненту для получения значения вероятности.

Пункты оценки:
1. совпадение предсказанных классов с оными у модели sklearn. Для проверки совпадения используйте функцию `assert_preds_correct`.
2. совпадение значений предсказанных вероятностей принадлежности классами с оными у модели sklearn. Значения вероятностей считаются равными, если функция `assert_probs_correct` возвращает True.

Сложность: математический гений.

In [243]:
x_train, y_train, x_test, y_test = load_data('multinomial')

In [244]:
class NaiveMultinomial(NaiveBayes):
    """
    Наивный Байесовский классификатор, реализующий Мультиномиальное
    распределение.
    """

    def __init__(self, alpha=1.0, n_classes=0):
        """
        Инициализирует Мультиномиальный Наивный Байес классификатор.

        Args:
            alpha (float, optional): Параметр сглаживания Лапласа.
            n_classes (int): Количество классов.
        """
        super().__init__(n_classes=n_classes)
        self.alpha = alpha
        self.n_samples = None
        self.n_features = None
        self.classes_ = None
        self._class_to_idx = {}
        self._y_indexed = None

    def compute_marginal_probability(self, x) -> float:
        """
        Вычисляет безусловную вероятность P(x) для заданного экземпляра x.

        P(x) = sum_k (P(x|C_k) * P(C_k)) по всем классам.

        Args:
            x (np.ndarray): Вектор признаков для одного экземпляра.

        Returns:
            float: Безусловная вероятность P(x).
        """
        marginal_prob = 0.0
        for c_idx in range(self.n_classes):
            marginal_prob += (self.compute_likelihood(x, c_idx) *
                              self.compute_prior(c_idx))
        return marginal_prob

    def compute_prior(self, c_idx: int) -> float:
        """
        Вычисляет априорную вероятность P(C_k) для заданного индекса класса.

        Args:
            c_idx (int): Индекс класса.

        Returns:
            float: Априорная вероятность P(C_k).
        """
        return self.params['prior'][c_idx]

    def compute_likelihood(self, x, c_idx: int) -> float:
        """
        Вычисляет вероятность правдоподобия P(x|C_k) для заданного вектора
        признаков x и класса C_k.

        Предполагается мультиномиальное распределение для признаков:
        P(x|C_k) = (N! / (x_1! * ... * x_k!)) * (p_1^x_1 * ... * p_k^x_k),
        где N = sum(x_i), x_i - количество i-го признака, p_i - условная
        вероятность i-го признака при условии класса C_k.
        Расчет выполняется в логарифмическом пространстве для предотвращения
        численных проблем, затем результат экспоненцируется.

        Args:
            x (np.ndarray): Одномерный массив признаков.
            c_idx (int): Индекс класса.

        Returns:
            float: Вычисленная вероятность правдоподобия P(x|C_k).
        """
        p_c = self.params['theta'][c_idx]
        N = np.sum(x)

        # Проверка, что N > 0, иначе lgamma(N+1) может быть некорректным
        if N == 0:
            # Если образец пуст, правдоподобие зависит только от p_c
            log_likelihood = 0.0
        else:
            log_likelihood = math.lgamma(N + 1)  # log(N!)

        for feature_count, prob_feature_given_class in zip(x, p_c):
            if feature_count > 0:
                # - log(x_i!)
                log_likelihood -= math.lgamma(feature_count + 1)
                if prob_feature_given_class <= 0:
                    log_likelihood = -np.inf
                    break
                # + x_i * log(p_i)
                log_likelihood += feature_count * np.log(
                    prob_feature_given_class
                )

        return np.exp(log_likelihood)

    # --- FITTING ---

    def _estimate_prior(self, y):
        """
        Оценивает априорные вероятности P(C_k) для каждого класса.

        P(C_k) вычисляется как отношение количества образцов класса C_k к
        общему количеству образцов в обучающем наборе.

        Args:
            y (np.ndarray): Целевые метки.
        """
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.n_classes = len(self.classes_)
        self.n_samples = len(y)
        self._class_to_idx = {cls: i for i, cls in enumerate(self.classes_)}
        y_indexed = np.array([self._class_to_idx[label] for label in y])
        self._y_indexed = y_indexed

        # Расчет априорных вероятностей
        prior_counts = Counter(y_indexed)
        self.params['prior'] = np.array([
            prior_counts[c_idx] / self.n_samples
            for c_idx in range(self.n_classes)
        ])

    def _estimate_params(self, x, y):
        """
        Оценивает условные вероятности P(f_i|C_k) для каждого признака f_i
        и класса C_k.

        Используется сглаживание Лапласа:
        P(f_i | C_k) = (count_of_f_i_in_C_k + alpha) /
                       (total_count_of_all_features_in_C_k +
                        alpha * n_features),
        где `f_i` соответствует отдельному признаку,
        `count_of_f_i_in_C_k` - количество вхождений этого признака в
        образцах класса C_k,
        `total_count_of_all_features_in_C_k` - общее количество всех
        признаков во всех образцах класса C_k.

        Args:
            x (np.ndarray): Обучающие данные.
            y (np.ndarray): Целевые метки.
        """
        x = np.asarray(x, dtype=np.intc)
        if x.ndim == 1:
            x = x.reshape(-1, 1)
        self.n_features = x.shape[1]

        y_indexed = self._y_indexed

        feature_counts_per_class = np.zeros(
            (self.n_classes, self.n_features), dtype=float
        )
        total_feature_counts_per_class = np.zeros(
            self.n_classes, dtype=float
        )

        for i in range(self.n_samples):
            c_idx = y_indexed[i]
            sample_features = x[i]
            feature_counts_per_class[c_idx] += sample_features
            total_feature_counts_per_class[c_idx] += np.sum(sample_features)

        self.params['theta'] = np.zeros(
            (self.n_classes, self.n_features), dtype=float
        )

        for c_idx in range(self.n_classes):
            denominator = (
                total_feature_counts_per_class[c_idx] +
                self.alpha * self.n_features
            )
            numerator = feature_counts_per_class[c_idx] + self.alpha
            self.params['theta'][c_idx] = numerator / denominator

In [245]:
# Определяем количество классов динамически
n_classes_from_data_m = len(np.unique(y_train))
print("Количество классов:", n_classes_from_data_m)

Количество классов: 2


In [246]:
# Создание и обучение модели
# Модель NaiveMultinomial собственная
model_multinomial = NaiveMultinomial(alpha=1.0, n_classes=n_classes_from_data_m)
model_multinomial.fit(x_train, y_train)
print("Модель NaiveMultinomial обучена.")

# Модель sklearn
sklearn_model_multinomial = MultinomialNB(alpha=1.0)
sklearn_model_multinomial.fit(x_train, y_train)
print("Модель MultinomialNB из sklearn обучена.")

Модель NaiveMultinomial обучена.
Модель MultinomialNB из sklearn обучена.


In [247]:
# Оценка качества модели
preds_multinomial = model_multinomial.predict(x_test, return_probs=False)
probs_multinomial = model_multinomial.predict(x_test, return_probs=True)

sklearn_preds_multinomial = sklearn_model_multinomial.predict(x_test)
sklearn_probs_multinomial = sklearn_model_multinomial.predict_proba(x_test)

# Метрики собственной модели Multinomial Naive Bayes
print_classification_metrics(
    y_test,
    preds_multinomial,
    title="Метрики качества собственной модели Multinomial Naive Bayes",
)

# Метрики модели Sklearn MultinomialNB
print_classification_metrics(
    y_test,
    sklearn_preds_multinomial,
    title="Метрики качества модели Sklearn MultinomialNB",
)


--- Метрики качества собственной модели Multinomial Naive Bayes ---
F1-мера: 0.8173
Recall: 0.8173
Precision: 0.8173

--- Метрики качества модели Sklearn MultinomialNB ---
F1-мера: 0.8173
Recall: 0.8173
Precision: 0.8173


In [248]:
# Сравните вашу модель с аналогом sklearn (MultinomialNB)
preds_multinomial_ok = assert_preds_correct(
    preds_multinomial, sklearn_preds_multinomial
)
print(
    "Проверка совпадения предсказаний:",
    preds_multinomial_ok
)

probs_multinomial_ok = assert_probs_correct(
    probs_multinomial, sklearn_probs_multinomial
)
print(
    "Проверка совпадения вероятностей:",
    probs_multinomial_ok
)

Проверка совпадения предсказаний: True
Проверка совпадения вероятностей: True


In [249]:
# Делаем окончательный вывод на основе всех результатов
if preds_multinomial_ok and probs_multinomial_ok:
    print("\nВсе проверки для Multinomial Naive Bayes пройдены успешно!")
else:
    print(
        "\nВнимание: Некоторые проверки для Multinomial Naive Bayes НЕ "
        "пройдены."
    )
    if not preds_multinomial_ok:
        print("  - Проверка предсказаний НЕ пройдена.")
    if not probs_multinomial_ok:
        print("  - Проверка вероятностей НЕ пройдена.")



Все проверки для Multinomial Naive Bayes пройдены успешно!
